# Recall — AI Agent with Memory (100% Free / No API Key Version)

Same idea — a voice/text assistant with long-term memory — but every paid piece is swapped
for a free, open-source equivalent:

| Piece | Paid version | Free version here |
|---|---|---|
| Chat model | OpenAI GPT-4o | Qwen2.5-1.5B-Instruct (local, open-source) |
| Speech-to-text | OpenAI Whisper API | `openai-whisper` package (same model, runs locally) |
| Text-to-speech | OpenAI TTS API | `edge-tts` (free, no account) |
| Memory embeddings | OpenAI embeddings | ChromaDB's built-in local embedding model |

**No API key, no billing, no signup required anywhere in this notebook.**

Trade-off: the local chat model is much smaller than GPT-4o, so answers will be noticeably
less capable — fine for testing the memory/voice pipeline, not a GPT-4o replacement.

**Before running:** In Colab, go to Runtime → Change runtime type → select a **T4 GPU**.
This will run painfully slowly on CPU-only.

## 1. Install dependencies (no API keys needed for any of these)

In [ ]:
!pip install -q transformers accelerate chromadb gradio openai-whisper edge-tts sentencepiece


## 2. (Optional) Mount Google Drive for persistent memory

Skip this if you're fine with memory resetting when the runtime disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHROMA_PATH = "/content/drive/MyDrive/recall_agent_free/chroma_db"
import os
os.makedirs(CHROMA_PATH, exist_ok=True)
print("Using persistent path:", CHROMA_PATH)


If you skipped Drive mounting, run this instead:

In [ ]:
# Only run this if you did NOT mount Drive above
# import os
# CHROMA_PATH = "/content/chroma_db"
# os.makedirs(CHROMA_PATH, exist_ok=True)


## 3. Load the local chat model

First run downloads ~3GB of model weights — takes a minute or two, then it's cached.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("Chat model loaded.")


## 4. Long-term memory layer (ChromaDB, local embeddings — no key needed)

In [ ]:
import uuid
import time
from typing import List

import chromadb

client = chromadb.PersistentClient(path=CHROMA_PATH)
# No embedding_function specified -> Chroma uses its built-in local
# sentence-transformers model automatically. No API key involved.
collection = client.get_or_create_collection(name="memory_default_user")


class LongTermMemory:
    def __init__(self, collection):
        self.collection = collection

    def add_exchange(self, user_message: str, assistant_message: str):
        doc_id = str(uuid.uuid4())
        text = f"User: {user_message}\nAssistant: {assistant_message}"
        self.collection.add(
            documents=[text],
            metadatas=[{"timestamp": time.time()}],
            ids=[doc_id],
        )

    def search(self, query: str, k: int = 5) -> List[str]:
        if self.collection.count() == 0:
            return []
        n = min(k, self.collection.count())
        results = self.collection.query(query_texts=[query], n_results=n)
        return results["documents"][0] if results["documents"] else []

    def get_context_block(self, query: str, k: int = 5) -> str:
        memories = self.search(query, k=k)
        if not memories:
            return ""
        joined = "\n---\n".join(memories)
        return f"Relevant memories from past conversations with this user:\n{joined}\n"


memory = LongTermMemory(collection)
print("Memory layer ready.")


## 5. The agent (local model + short-term + long-term memory)

In [ ]:
SYSTEM_PROMPT = """You are a helpful personal AI assistant with long-term memory.
You remember details about the user across conversations and refer back to them
naturally. If relevant memories are provided below, use them to personalize your
response. If none are relevant, ignore them. Keep answers reasonably concise."""


class Agent:
    def __init__(self, memory: LongTermMemory):
        self.memory = memory
        self.history = []  # short-term session buffer, list of {"role", "content"}

    def chat(self, user_message: str) -> str:
        memory_block = self.memory.get_context_block(user_message, k=5)
        system_content = SYSTEM_PROMPT
        if memory_block:
            system_content += f"\n\n{memory_block}"

        messages = [{"role": "system", "content": system_content}]
        messages += self.history[-12:]
        messages.append({"role": "user", "content": user_message})

        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        reply = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        self.history.append({"role": "user", "content": user_message})
        self.history.append({"role": "assistant", "content": reply})
        self.memory.add_exchange(user_message, reply)
        return reply


agent = Agent(memory)
print("Agent ready.")


## 6. Voice helpers — local Whisper (STT) + edge-tts (TTS), no keys

In [ ]:
import whisper
import edge_tts
import asyncio

whisper_model = whisper.load_model("base")  # "tiny" is faster/less accurate, "small" more accurate/slower

def transcribe(audio_path: str) -> str:
    result = whisper_model.transcribe(audio_path)
    return result["text"].strip()

async def _synthesize_async(text: str, out_path: str):
    communicate = edge_tts.Communicate(text, voice="en-US-AriaNeural")
    await communicate.save(out_path)

def synthesize(text: str, out_path: str = "/content/reply.mp3") -> str:
    asyncio.run(_synthesize_async(text, out_path))
    return out_path

print("Voice helpers ready.")


## 7. Gradio UI

Same interface as the paid version — text tab and voice tab, both backed by the free local stack.

In [ ]:
import gradio as gr

def handle_text(message, history):
    return agent.chat(message)

def handle_voice(audio_path):
    if audio_path is None:
        return "", None, None
    user_text = transcribe(audio_path)
    reply_text = agent.chat(user_text)
    reply_audio = synthesize(reply_text)
    return user_text, reply_text, reply_audio

with gr.Blocks(title="Recall — Free AI Agent with Memory") as demo:
    gr.Markdown("# Recall (Free version)\nLocal model + local memory + local voice. No API key.")

    with gr.Tab("Text chat"):
        gr.ChatInterface(fn=handle_text, type="messages")

    with gr.Tab("Voice chat"):
        mic = gr.Audio(sources=["microphone"], type="filepath", label="Record")
        transcript_box = gr.Textbox(label="Heard")
        reply_box = gr.Textbox(label="Reply")
        reply_audio = gr.Audio(label="Reply (spoken)", autoplay=True)
        mic.change(fn=handle_voice, inputs=mic,
                   outputs=[transcript_box, reply_box, reply_audio])

demo.launch(share=True, debug=True)
